In [1]:
import torch.nn as nn
import torch

class LinearLayer(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.is_bias = bias
        self.weight = torch.nn.Parameter(torch.rand(self.in_features, self.out_features))

        if self.is_bias:
            self.bias = torch.nn.Parameter(torch.rand(self.out_features))
    
    def forward(self, x: torch.tensor):
        output = x @ self.weight
        if self.is_bias:
            output += self.bias
        return output

class LayerNorm(nn.Module):
    def __init__(self, in_features: int, eps: float = 0.001):
        super().__init__()
        self.in_features = in_features
        self.gamma = torch.nn.Parameter(torch.ones(self.in_features))
        self.beta = torch.nn.Parameter(torch.zeros(self.in_features))
        self.eps = eps
    
    def forward(self, x: torch.tensor):
        x_mean = x.mean(dim = -1, keepdim = True)
        x_var = x.var(dim = -1, keepdim = True, unbiased = False)
        x_norm = (x - x_mean) / torch.sqrt(x_var + self.eps)
        return x_norm * self.gamma + self.beta

class SingleHeadAttention(nn.Module):
    def __init__(self, d_model: int, d_q: int, d_v: int):
        super().__init__()

        self.W_q = torch.nn.Parameter(torch.rand(d_model, d_q))
        self.W_k = torch.nn.Parameter(torch.rand(d_model, d_q))
        self.W_v = torch.nn.Parameter(torch.rand(d_model, d_v))

        self.d_q = d_q
        self.d_v = d_v
    
    def _compute_attention_scores(self, Query: torch.tensor, Key: torch.tensor, Value: torch.tensor):
        d_k = self.d_q
        A = Query @ Key.transpose(-1, -2)
        adj_A = torch.softmax((A / (d_k ** 0.5)), dim = -1)
        Z = adj_A @ Value
        return Z
    
    def forward(self, x: torch.tensor):
        Query = x @ self.W_q
        Key = x @ self.W_k
        Value = x @ self.W_v
        return self._compute_attention_scores(Query, Key, Value)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, h: int):
        super().__init__()
        
        assert d_model % h == 0, "d_model must be Divisible by h"
        self.d_model = d_model
        self.h = h
        self.d_k = self.d_model // self.h
        self.d_v = self.d_k

        self.W_q = torch.nn.Parameter(torch.rand(d_model, h * self.d_k))
        self.W_k = torch.nn.Parameter(torch.rand(d_model, h * self.d_k))
        self.W_v = torch.nn.Parameter(torch.rand(d_model, h * self.d_v))

        # Linear projection for Converting from concatinated d_model to real d_mdoel, mixing the learned features
        self.linear_projection = LinearLayer(in_features = self.d_model, out_features = self.d_model)

    def _compute_attention_scores(self, Query: torch.tensor, Key: torch.tensor, Value: torch.tensor):
        # B, h, T, d_q = Query.shape
        # B, h, T, d_q = Key.shape
        A = Query @ Key.transpose(-1, -2) # A.shape = (B, h, T, d_q) @ (B, h, T, d_q).T = (B, h, T, d_q) @ (B, h, d_q, T) --> (B, h, T, T)
        adj_A = torch.softmax((A / (self.d_k ** 0.5)), dim = -1)
        Z = adj_A @ Value
        return Z
    
    def forward(self, x: torch.tensor):
        B, T, d_model = x.shape
        query = x @ self.W_q # query.shape = (B, T, d_model) @ (d_model, d_model) = (B, T, d_model) --> (B, T, h, d_k) --> (B, h, T, d_k)
        key = x @ self.W_k
        value = x @ self.W_v

        query = query.reshape(B, T, self.h, self.d_k).transpose(1, 2)
        key = key.reshape(B, T, self.h, self.d_k).transpose(1, 2)
        value = value.reshape(B, T, self.h, self.d_v).transpose(1, 2)

        Z = self._compute_attention_scores(query, key, value)
        return self.linear_projection(Z.transpose(1, 2).contiguous().view(B, T, d_model))


class PositionalEmbeddings(nn.Module):
    def __init__(self):
        super().__init__()

class InputEmbeddings(nn.Module):
    def __init__(self):
        super().__init__()

class SkipConnections(nn.Module):
    def __init__(self):
        super().__init__()

class FeedForward(nn.Module):
    def __init__(self, d_ff: int, d_model: int):
        super().__init__()
        self.layer_1 = LinearLayer(d_model, d_ff)
        self.layer_2 = LinearLayer(d_ff, d_ff)
    
    def forward(self, x: torch.tensor):
        return self.layer_2(torch.nn.ReLU()(self.layer_1(x)))

In [2]:
context = "My name is Himanshu Singh"
context.split()

['My', 'name', 'is', 'Himanshu', 'Singh']

In [3]:
my_dict = {
    "My": 10,
    "name": 16,
    "is": 18,
    "Himanshu": 23,
    "Singh": 12
}

In [4]:
token_id = []
for c in context.split(" "):
    token_id.append(my_dict[c])

In [5]:
token_id

[10, 16, 18, 23, 12]

In [6]:
vocab = 27
d_model = 512
embeddings = torch.rand(vocab, d_model)
x = embeddings[token_id]

In [7]:
x = x.unsqueeze(0)

In [8]:
x.shape

torch.Size([1, 5, 512])

In [9]:
h = 8
batch_size = 1
seq_len = len(context.split())
d_model = 512
d_q = d_model // h

In [10]:
MHA = MultiHeadAttention(d_model = d_model, h = h)
MHA(x).shape

torch.Size([1, 5, 512])